# 🧠 Data Mining — Classification: Advanced Techniques II
## Lecture 3 Applied Notebook
**Haydar Kılıç | Artificial Intelligence Engineering | Spring Semester**

---
### 📚 Notebook Contents

| # | Section | Topic |
|---|---------|-------|
| 1 | ANN In-Depth | Biological Neuron, Forward/Backward Propagation, Mini-batch |
| 2 | SVM | Linear SVM, Dual Problem, Kernel Trick |
| 3 | Bayesian Networks | DAG, Conditional Independence, CPT |
| 4 | Ensemble Learning | Bagging, AdaBoost, Bias-Variance |
| 5 | Class Imbalance | Undersampling, Oversampling, SMOTE |
| 6 | Performance Metrics | Confusion Matrix, ROC, PR Curve |


In [ ]:
# ============================================================
# SETUP AND IMPORTS
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
import warnings
warnings.filterwarnings('ignore')

# Sklearn
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, AdaBoostClassifier, RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (accuracy_score, confusion_matrix,
                              classification_report, roc_curve, auc,
                              precision_recall_curve, f1_score,
                              precision_score, recall_score)
from sklearn.datasets import make_classification, make_circles, make_moons, load_breast_cancer
from scipy.special import expit as sigmoid
from scipy.stats import norm

print('✅ All libraries loaded successfully!')
print(f'NumPy: {np.__version__} | Pandas: {pd.__version__}')


---
# SECTION 1: Artificial Neural Networks (ANN) — In-Depth

## 📖 Biological → Artificial Neuron Analogy

| Biological | Artificial Counterpart |
|------------|------------------------|
| Dendrite | Input (x₁, x₂, ..., xₙ) |
| Synapse strength | Weight (w₁, w₂, ..., wₙ) |
| Cell body (soma) | Summation function (Σ) |
| Axon | Activation output |
| Firing threshold | Bias (b) + Activation function |

## Single Neuron Mathematics
$$z = \\sum_{i=1}^{n} w_i x_i + b = \\mathbf{w}^T\\mathbf{x} + b$$
$$\\hat{y} = f(z) \\quad \\text{(activation function)}$$

## Multi-Layer Network — Layer l, Neuron i
$$a_i^l = f(z_i^l) = f\\left(\\sum_j w_{ij}^l a_j^{l-1} + b_i^l\\right)$$


In [ ]:
# ============================================================
# SINGLE NEURON: FORWARD PASS STEP BY STEP
# ============================================================
print('=' * 60)
print('SINGLE NEURON — FORWARD PASS SIMULATION')
print('=' * 60)

# Parameters
x = np.array([2.0, 3.0, -1.0])  # inputs
w = np.array([0.4, -0.2, 0.7])  # weights
b = 0.1                           # bias

# Summation
z = np.dot(w, x) + b
print(f'\nInputs      : x = {x}')
print(f'Weights     : w = {w}')
print(f'Bias        : b = {b}')
print(f'\nSummation z = w₁x₁ + w₂x₂ + w₃x₃ + b')
print(f'           = ({w[0]}×{x[0]}) + ({w[1]}×{x[1]}) + ({w[2]}×{x[2]}) + {b}')
print(f'           = {w[0]*x[0]:.1f} + {w[1]*x[1]:.1f} + {w[2]*x[2]:.1f} + {b}')
print(f'           = {z:.4f}')

# Different activation functions
activations = {
    'Sigmoid σ(z)' : 1 / (1 + np.exp(-z)),
    'Tanh'         : np.tanh(z),
    'ReLU'         : max(0, z),
    'Sign (±1)'    : np.sign(z)
}
print("{:<20} {}".format("Activation", "Output"))
print('-' * 35)
for name, value in activations.items():
    print(f'{name:<20} {value:.6f}')


## 1.1 Slide Mini-batch Example — Step by Step

**Model:** Single neuron → $z = wx + b$, $\\hat{y} = \\sigma(z)$

**Loss:** $\\text{Loss}_k = \\frac{1}{2}(y_k - \\hat{y}_k)^2$

**Initialization:** $w^{(0)} = 0.5$, $b^{(0)} = 0$, $\\eta = 0.1$

**Mini-batch:** $(x_1,y_1)=(0,0)$, $(x_2,y_2)=(1,1)$, $(x_3,y_3)=(2,1)$


In [ ]:
# ============================================================
# SLIDE MINI-BATCH EXAMPLE — FULL IMPLEMENTATION
# ============================================================
def sigmoid_fn(z):
    return 1 / (1 + np.exp(-z))

# Data and initialization
data = [(0, 0), (1, 1), (2, 1)]
w, b = 0.5, 0.0
eta  = 0.1

print('=' * 65)
print('MINI-BATCH TRAINING — SLIDE EXAMPLE')
print('=' * 65)
print(f'Initialization: w={w}, b={b}, η={eta}')

# === FORWARD PASS ===
print('\n📌 STEP 1: FORWARD PASS')
print("{:<5} {:<6} {:<6} {:<14} {:<14} {}".format("k", "xₖ", "yₖ", "zₖ=wxₖ+b", "ŷₖ=σ(zₖ)", "Lossₖ"))
print('-' * 60)

predictions = []
losses      = []
for k, (x_k, y_k) in enumerate(data, 1):
    z_k    = w * x_k + b
    yhat_k = sigmoid_fn(z_k)
    loss_k = 0.5 * (y_k - yhat_k) ** 2
    predictions.append(yhat_k)
    losses.append(loss_k)
    print(f'{k:<5} {x_k:<6} {y_k:<6} {z_k:<14.4f} {yhat_k:<14.4f} {loss_k:.4f}')

print(f'\n  Mini-batch Total Loss = {sum(losses):.4f}')
print(f'  (Slide reference: ≈ 0.232 ✅)')

# === BACKWARD PASS ===
print('\n📌 STEP 2: BACKWARD PASS (Gradient Computation)')
print("   Sigmoid derivative: σ'(z) = ŷ(1-ŷ)")
print('   ∂Loss/∂w = (ŷ-y)·σ\'(z)·x    |    ∂Loss/∂b = (ŷ-y)·σ\'(z)')
print()
print("{:<5} {:<9} {:<12} {:<12} {}".format("k", "ŷₖ", "σ'(zₖ)", "∂L/∂b", "∂L/∂w"))
print('-' * 55)

dL_dw_total = 0.0
dL_db_total = 0.0
for k, (x_k, y_k) in enumerate(data, 1):
    yhat_k       = predictions[k-1]
    z_k          = w * x_k + b
    sigma_prime  = yhat_k * (1 - yhat_k)
    dL_db_k      = (yhat_k - y_k) * sigma_prime
    dL_dw_k      = dL_db_k * x_k
    dL_dw_total += dL_dw_k
    dL_db_total += dL_db_k
    print(f'{k:<5} {yhat_k:<9.4f} {sigma_prime:<12.4f} {dL_db_k:<12.4f} {dL_dw_k:.4f}')

print(f'\n  ΔW = {dL_dw_total:.4f}  (Slide reference: ≈ -0.195)')
print(f'  Δb = {dL_db_total:.4f}  (Slide reference: ≈ -0.017)')

# === UPDATE ===
print('\n📌 STEP 3: PARAMETER UPDATE (Gradient Descent)')
w_new = w - eta * dL_dw_total
b_new = b - eta * dL_db_total
print(f'  w_new = {w} - {eta}×({dL_dw_total:.4f}) = {w_new:.4f}')
print(f'  b_new = {b} - {eta}×({dL_db_total:.4f}) = {b_new:.4f}')
print(f'\n  (w, b): ({w}, {b}) ──▶ ({w_new:.4f}, {b_new:.4f})')
print(f'  Slide: (0.5, 0) → (0.5195, 0.0017) ✅')


In [ ]:
# ============================================================
# BACKWARD PASS — LOSS AND GRADIENT VISUALIZATION
# ============================================================

# Multi-epoch simulation
w_sim, b_sim = 0.5, 0.0
eta_sim = 0.1
n_epoch = 200

loss_history = []
w_history, b_history = [w_sim], [b_sim]

for epoch in range(n_epoch):
    dw_total = 0.0
    db_total = 0.0
    total_loss = 0.0
    for x_k, y_k in data:
        z_k      = w_sim * x_k + b_sim
        yhat_k   = sigmoid_fn(z_k)
        sp       = yhat_k * (1 - yhat_k)
        total_loss += 0.5 * (y_k - yhat_k)**2
        dw_total += (yhat_k - y_k) * sp * x_k
        db_total += (yhat_k - y_k) * sp
    w_sim -= eta_sim * dw_total
    b_sim -= eta_sim * db_total
    loss_history.append(total_loss)
    w_history.append(w_sim)
    b_history.append(b_sim)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Mini-batch Training: Learning Process Visualization', fontsize=13, fontweight='bold')

# Loss curve
axes[0].plot(loss_history, 'b-', linewidth=2)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Total Loss')
axes[0].set_title('Loss Curve'); axes[0].grid(True, alpha=0.3)
axes[0].axhline(loss_history[-1], color='red', linestyle='--',
                label=f'Final loss: {loss_history[-1]:.4f}')
axes[0].legend()

# Parameter evolution
axes[1].plot(w_history, 'g-', linewidth=2, label='w')
axes[1].plot(b_history, 'r-', linewidth=2, label='b')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Parameter Value')
axes[1].set_title('Parameter Evolution'); axes[1].grid(True, alpha=0.3)
axes[1].legend(); axes[1].axvline(0, color='gray', alpha=0.5)

# Prediction vs ground truth (final model)
x_plot = np.linspace(-0.5, 2.5, 100)
y_plot = sigmoid_fn(w_sim * x_plot + b_sim)
axes[2].plot(x_plot, y_plot, 'b-', linewidth=2, label='Model Prediction')
for x_k, y_k in data:
    axes[2].scatter(x_k, y_k, c='red', s=100, zorder=5)
axes[2].scatter([], [], c='red', s=100, label='Ground Truth')
axes[2].axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='Threshold=0.5')
axes[2].set_xlabel('x'); axes[2].set_ylabel('σ(wx+b)')
axes[2].set_title(f'Final Model (w={w_sim:.3f}, b={b_sim:.3f})')
axes[2].grid(True, alpha=0.3); axes[2].legend()

plt.tight_layout()
plt.show()
print(f'\n✅ After {n_epoch} epochs: w={w_sim:.4f}, b={b_sim:.4f}')


In [ ]:
# ============================================================
# MULTI-LAYER NETWORK — FORWARD PASS FROM SCRATCH
# Architecture: 2 → 3 → 1  (Input → Hidden → Output)
# ============================================================

np.random.seed(42)

# Weights (random initialization)
W1 = np.random.randn(3, 2) * 0.5   # Hidden layer: 3 neurons, 2 inputs
b1 = np.zeros((3, 1))
W2 = np.random.randn(1, 3) * 0.5   # Output layer: 1 neuron, 3 inputs
b2 = np.zeros((1, 1))

def sigmoid(z): return 1 / (1 + np.exp(-z))
def relu(z):    return np.maximum(0, z)

# Sample input
x_input = np.array([[1.5], [-0.8]])  # (2, 1)

print('=== MULTI-LAYER NETWORK: FORWARD PASS (2→3→1) ===')
print(f'Input x = {x_input.flatten()}')
print(f'\nW1 (3×2):\n{W1.round(3)}')
print(f'b1 (3×1): {b1.flatten()}')

# Layer 1 (Hidden)
z1 = W1 @ x_input + b1                # (3,1)
a1 = sigmoid(z1)                       # (3,1)
print(f'\n📌 Hidden Layer (Layer 1):')
print(f'  z1 = W1·x + b1 = {z1.flatten().round(4)}')
print(f'  a1 = σ(z1)     = {a1.flatten().round(4)}')

# Layer 2 (Output)
z2 = W2 @ a1 + b2                     # (1,1)
a2 = sigmoid(z2)                       # (1,1)
print(f'\n📌 Output Layer (Layer 2):')
print(f'  z2 = W2·a1 + b2 = {z2.flatten().round(4)}')
print(f'  a2 = σ(z2)      = {a2.flatten().round(4)}')
print(f'\n🎯 Prediction: ŷ = {a2[0,0]:.4f} → Class: {"+1" if a2[0,0]>0.5 else "-1"}')

# Network diagram
fig, ax = plt.subplots(figsize=(9, 6))
ax.set_xlim(0, 4); ax.set_ylim(0, 5)
ax.axis('off')
ax.set_title('Multi-Layer Neural Network: 2→3→1', fontsize=13, fontweight='bold')

layers = {
    'Input (Layer 0)':  ([0.5], [1.5, 3.5], 'lightblue'),
    'Hidden (Layer 1)': ([2.0], [1.0, 2.5, 4.0], 'lightyellow'),
    'Output (Layer 2)': ([3.5], [2.5], 'lightgreen')
}

positions = {}
for layer_name, (xs, ys, color) in layers.items():
    x_pos = xs[0]
    for i, y_pos in enumerate(ys):
        circle = plt.Circle((x_pos, y_pos), 0.3, color=color, ec='black', linewidth=1.5, zorder=5)
        ax.add_patch(circle)
        tag = layer_name.split()[0]
        if tag == 'Input':
            ax.text(x_pos, y_pos, f'x{i+1}', ha='center', va='center', fontsize=9, zorder=6)
            ax.text(x_pos, y_pos-0.55, f'{x_input[i,0]:.1f}', ha='center', va='top', fontsize=7, color='navy')
        elif tag == 'Hidden':
            ax.text(x_pos, y_pos, f'h{i+1}', ha='center', va='center', fontsize=9, zorder=6)
            ax.text(x_pos, y_pos-0.55, f'{a1[i,0]:.3f}', ha='center', va='top', fontsize=7, color='darkred')
        else:
            ax.text(x_pos, y_pos, 'ŷ', ha='center', va='center', fontsize=11, zorder=6)
            ax.text(x_pos, y_pos-0.55, f'{a2[0,0]:.3f}', ha='center', va='top', fontsize=7, color='darkgreen')
        positions[(layer_name, i)] = (x_pos, y_pos)

# Connections (input→hidden)
for i in range(2):
    for j in range(3):
        x0, y0 = 0.5, [1.5, 3.5][i]
        x1, y1 = 2.0, [1.0, 2.5, 4.0][j]
        ax.annotate('', xy=(x1-0.3, y1), xytext=(x0+0.3, y0),
                    arrowprops=dict(arrowstyle='->', color='gray', lw=0.8))

# Connections (hidden→output)
for j in range(3):
    x0, y0 = 2.0, [1.0, 2.5, 4.0][j]
    ax.annotate('', xy=(3.2, 2.5), xytext=(x0+0.3, y0),
                arrowprops=dict(arrowstyle='->', color='gray', lw=0.8))

for x_pos, label in [(0.5, 'Input\nLayer'), (2.0, 'Hidden\nLayer'), (3.5, 'Output\nLayer')]:
    ax.text(x_pos, 0.1, label, ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.text(1.1, 4.5, 'Sigmoid activation →', fontsize=8, color='gray', style='italic')
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# SKLEARN MLP — ARCHITECTURE COMPARISON
# ============================================================
from sklearn.datasets import load_iris
iris = load_iris()
X_iris, y_iris = iris.data, iris.target
X_tr, X_te, y_tr, y_te = train_test_split(X_iris, y_iris, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)

print('=== MLP ARCHITECTURE COMPARISON (Iris Dataset) ===')
print("\n{:<30} {:<12} {:<8} {}".format("Architecture", "Activation", "Epochs", "Test Accuracy"))
print('─' * 65)

results_mlp = []
for arch in [(10,), (50,), (100,), (50, 25), (100, 50), (100, 50, 25)]:
    for activation in ['relu', 'tanh', 'logistic']:
        mlp = MLPClassifier(hidden_layer_sizes=arch, activation=activation,
                            max_iter=1000, random_state=42)
        mlp.fit(X_tr_s, y_tr)
        acc = accuracy_score(y_te, mlp.predict(X_te_s))
        results_mlp.append({'Architecture': str(arch), 'Activation': activation,
                            'Epochs': mlp.n_iter_, 'Accuracy': acc})

df_mlp = pd.DataFrame(results_mlp)

# Show top 5 combinations
top5 = df_mlp.nlargest(5, 'Accuracy')
for _, row in top5.iterrows():
    print(f"{row['Architecture']:<30} {row['Activation']:<12} {int(row['Epochs']):<8} {row['Accuracy']:.4f}")

print(f'\n💡 Test samples: {len(y_te)} | Classes: 3 (Iris species)')

# Visualize
fig, ax = plt.subplots(figsize=(10, 5))
for act in ['relu', 'tanh', 'logistic']:
    df_f = df_mlp[df_mlp['Activation'] == act].reset_index(drop=True)
    ax.plot(range(len(df_f)), df_f['Accuracy'], 'o-', label=act, linewidth=2, markersize=8)

ax.set_xticks(range(6))
ax.set_xticklabels(['(10,)', '(50,)', '(100,)', '(50,25)', '(100,50)', '(100,50,25)'], rotation=20)
ax.set_ylabel('Test Accuracy'); ax.set_xlabel('Architecture')
ax.set_title('MLP: Architecture × Activation Comparison', fontsize=13, fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3); ax.set_ylim(0.8, 1.05)
plt.tight_layout()
plt.show()


---
# SECTION 2: Support Vector Machines (SVM)

## 📖 Theoretical Summary

**Primal Problem:** $\\min_{w,b} \\frac{\\|w\\|^2}{2}$ subject to $y_i(w^T x_i + b) \\geq 1$

**Dual Problem:** $\\max_\\lambda \\sum_i \\lambda_i - \\frac{1}{2}\\sum_{i,j} \\lambda_i \\lambda_j y_i y_j x_i^T x_j$

**Margin Width:** $\\frac{2}{\\|w\\|}$

**KKT Condition:** $\\lambda_i[y_i(w^T x_i + b) - 1] = 0$ → λ > 0 points are Support Vectors

## 2.1 Slide SVM Example — Manual Solution

**Data:**
- $x_1=(2,2), y_1=+1$ → $x_2=(4,4), y_2=+1$
- $x_3=(2,0), y_3=-1$ → $x_4=(0,2), y_4=-1$

**Solution:** $w=(1,1)$, $b=-3$, Decision boundary: $x_1+x_2=3$, Margin: $\\sqrt{2}$


In [ ]:
# ============================================================
# SLIDE SVM EXAMPLE — MANUAL VERIFICATION
# ============================================================
from sklearn.svm import SVC

# 4-point example from the slide
X_svm4 = np.array([[2,2],[4,4],[2,0],[0,2]], dtype=float)
y_svm4 = np.array([1, 1, -1, -1])

print('=== SLIDE SVM EXAMPLE ===')
print('Training Dataset:')
for xi, yi in zip(X_svm4, y_svm4):
    print(f'  x={xi}, y={yi:+d}')

# Solve with sklearn
svm4 = SVC(kernel='linear', C=1e6)
svm4.fit(X_svm4, y_svm4)

w_sol = svm4.coef_[0]
b_sol = svm4.intercept_[0]
margin = 2.0 / np.linalg.norm(w_sol)

print(f'\n📐 SOLUTION:')
print(f'  w = [{w_sol[0]:.4f}, {w_sol[1]:.4f}]   (Slide: [1, 1])')
print(f'  b = {b_sol:.4f}              (Slide: -3)')
print(f'  Margin width = 2/||w|| = {margin:.4f}  (Slide: √2 ≈ 1.4142)')
print(f'  Decision boundary: {w_sol[0]:.2f}·x₁ + {w_sol[1]:.2f}·x₂ + {b_sol:.2f} = 0')
print(f'  → x₁ + x₂ = 3 ✅')

print(f'\n🔵 Support Vectors:')
for sv in svm4.support_vectors_:
    f_val = w_sol @ sv + b_sol
    print(f'  {sv} → f(x)={f_val:.4f}')

# Check KKT conditions
print(f'\n✅ KKT Conditions (λᵢ[yᵢ(w·xᵢ+b) - 1] = 0):')
for xi, yi in zip(X_svm4, y_svm4):
    func = yi * (w_sol @ xi + b_sol) - 1
    status = 'Support Vector ✓' if abs(func) < 0.01 else 'Interior point'
    print(f'  x={xi}, y={yi:+d}: yᵢ(w·xᵢ+b)-1 = {func:.4f} → {status}')


In [ ]:
# ============================================================
# SVM VISUALIZATION — MARGIN AND SUPPORT VECTORS
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('SVM: Maximum Margin Hyperplane', fontsize=13, fontweight='bold')

# Left: 4-point example
ax = axes[0]
x1_r = np.linspace(-0.5, 5, 200)

def svm_line(x1, w, b, offset=0):
    return (-w[0]*x1 - b + offset) / w[1]

ax.plot(x1_r, svm_line(x1_r, w_sol, b_sol),     'k-',  lw=2.5, label='Decision Boundary')
ax.plot(x1_r, svm_line(x1_r, w_sol, b_sol,  1),  'b--', lw=1.5, label='Margin +1')
ax.plot(x1_r, svm_line(x1_r, w_sol, b_sol, -1),  'r--', lw=1.5, label='Margin -1')

y_up   = svm_line(x1_r, w_sol, b_sol,  1)
y_down = svm_line(x1_r, w_sol, b_sol, -1)
ax.fill_between(x1_r, y_down, y_up, alpha=0.1, color='yellow', label='Margin Region')

for xi, yi in zip(X_svm4, y_svm4):
    c = '#2ecc71' if yi==1 else '#e74c3c'
    m = 's' if yi==1 else 'o'
    ax.scatter(*xi, c=c, marker=m, s=150, edgecolors='black', zorder=5)

for sv in svm4.support_vectors_:
    ax.add_patch(plt.Circle(sv, 0.25, color='blue', fill=False, lw=2.5, zorder=6))

ax.set_xlim(-0.5, 5); ax.set_ylim(-0.5, 5)
ax.set_xlabel('x₁'); ax.set_ylabel('x₂')
ax.set_title('4-Point Slide Example')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# Margin width arrow
ax.annotate('', xy=(1.5, 1.5), xytext=(2.5, 0.5),
            arrowprops=dict(arrowstyle='<->', color='purple', lw=2))
ax.text(1.5, 0.8, f'Margin={margin:.3f}\n(=√2)', color='purple', fontsize=8)

# Right: Larger dataset
ax2 = axes[1]
np.random.seed(42)
X_large = np.r_[np.random.randn(30, 2) + [2, 2], np.random.randn(30, 2) + [-2, -2]]
y_large = np.r_[np.ones(30), -np.ones(30)]

svm_lin = SVC(kernel='linear', C=1.0)
svm_lin.fit(X_large, y_large)

xx, yy = np.meshgrid(np.linspace(-5, 6, 200), np.linspace(-5, 6, 200))
Z = svm_lin.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
ax2.contourf(xx, yy, Z, alpha=0.25, cmap=ListedColormap(['#fadbd8', '#d5f5e3']))
ax2.contour(xx, yy, Z, colors='black', linewidths=1)

for xi, yi in zip(X_large, y_large):
    c = '#27ae60' if yi==1 else '#c0392b'
    ax2.scatter(*xi, c=c, s=40, alpha=0.7)
for sv in svm_lin.support_vectors_:
    ax2.add_patch(plt.Circle(sv, 0.2, color='blue', fill=False, lw=2, zorder=6))

ax2.set_xlim(-5, 6); ax2.set_ylim(-5, 6)
ax2.set_xlabel('x₁'); ax2.set_ylabel('x₂')
ax2.set_title(f'Large Dataset (Support Vectors: {len(svm_lin.support_vectors_)})')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# KERNEL TRICK — FEATURE SPACE TRANSFORMATION
# ============================================================
# Slide example: φ: (x1,x2) → (x1²-x1, x2²-x2)

np.random.seed(42)
X_nl, y_nl = make_circles(n_samples=150, noise=0.12, factor=0.45, random_state=42)
y_nl_svm   = np.where(y_nl==0, -1, 1)

# Apply transformation to original space
def phi_transform(X):
    return np.column_stack([X[:,0]**2 - X[:,0],
                            X[:,1]**2 - X[:,1]])

X_phi = phi_transform(X_nl)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Kernel Trick: Non-Linear Feature Transformation', fontsize=13, fontweight='bold')

colors = ['#c0392b' if yi==-1 else '#27ae60' for yi in y_nl_svm]

# 1. Original space (not solvable)
ax = axes[0]
ax.scatter(X_nl[:,0], X_nl[:,1], c=colors, s=40, alpha=0.8)
ax.set_title('Feature Space (Not Solvable ❌)', fontweight='bold')
ax.set_xlabel('x₁'); ax.set_ylabel('x₂'); ax.grid(True, alpha=0.3)

# 2. Transformed space
ax = axes[1]
ax.scatter(X_phi[:,0], X_phi[:,1], c=colors, s=40, alpha=0.8)
ax.set_title('φ(X) Space: (x₁²-x₁, x₂²-x₂)', fontweight='bold')
ax.set_xlabel('x₁²-x₁'); ax.set_ylabel('x₂²-x₂'); ax.grid(True, alpha=0.3)
ax.set_xlim(-0.3, 0.3); ax.set_ylim(-0.3, 0.3)

# 3. Kernel SVM (RBF) — boundary
ax = axes[2]
svm_rbf = SVC(kernel='rbf', gamma=2.0, C=1.0)
svm_rbf.fit(X_nl, y_nl_svm)
xx, yy = np.meshgrid(np.linspace(-1.5, 1.5, 200), np.linspace(-1.5, 1.5, 200))
Z = svm_rbf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
ax.contourf(xx, yy, Z, alpha=0.3, cmap=ListedColormap(['#fadbd8', '#d5f5e3']))
ax.contour(xx, yy, Z, colors='black', linewidths=1)
ax.scatter(X_nl[:,0], X_nl[:,1], c=colors, s=40, alpha=0.8)
for sv in svm_rbf.support_vectors_:
    ax.add_patch(plt.Circle(sv, 0.07, color='blue', fill=False, lw=1.5, zorder=6))
acc_rbf = accuracy_score(y_nl_svm, svm_rbf.predict(X_nl))
ax.set_title(f'RBF Kernel SVM (Accuracy: {acc_rbf:.1%}) ✅', fontweight='bold')
ax.set_xlabel('x₁'); ax.set_ylabel('x₂'); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print('💡 Kernel Trick: Use K(u,v) = <φ(u),φ(v)> without computing φ(x) explicitly!')
print('   → Polynomial: K(u,v) = (uᵀv+1)ᵖ')
print('   → RBF       : K(u,v) = exp(-||u-v||²/2σ²)')
print('   → Sigmoid   : K(u,v) = tanh(kuᵀv - δ)')


---
# SECTION 3: Bayesian Networks

## 📖 Theoretical Summary

**DAG:** Directed Acyclic Graph — each node is a random variable

**Local Markov Property:**
$$X_i \\perp \\text{NonDesc}(X_i) \\mid pa(X_i)$$

**Joint Distribution (Chain Rule):**
$$P(X) = \\prod_{i=1}^{d} P(X_i \\mid pa(X_i))$$

**Heart Disease Example (Slide):**
$$P(X) = P(Ex)\\cdot P(Di)\\cdot P(HD|Ex,Di)\\cdot P(HB|Di)\\cdot P(BP|HD)\\cdot P(CP|HD,HB)$$


In [ ]:
# ============================================================
# SLIDE HEART DISEASE BAYESIAN NETWORK
# Conditional Probability Tables (CPT) + Inference
# ============================================================

print('=== HEART DISEASE BAYESIAN NETWORK ===')
print('Structure: Exercise(Ex), Diet(Di) → Heart Disease(HD)')
print('           Diet(Di)  → Heartburn(HB)')
print('           Heart Disease(HD) → Blood Pressure(BP)')
print('           Heart Disease(HD) + Heartburn(HB) → Chest Pain(CP)')

# =================================
# PRIOR PROBABILITIES (root nodes)
# =================================
P_Ex = {True: 0.55, False: 0.45}           # Regular exercise?
P_Di = {True: 0.40, False: 0.60}           # Healthy diet?

# =================================
# CONDITIONAL PROBABILITY TABLES (CPT)
# =================================
# P(HD | Ex, Di)
P_HD_gEx_Di = {
    (True,  True):  0.10,   # exercise + diet → low risk
    (True,  False): 0.25,   # exercise but poor diet
    (False, True):  0.35,   # good diet but no exercise
    (False, False): 0.60,   # neither exercise nor diet → high risk
}

# P(HB | Di)
P_HB_gDi = {True: 0.20, False: 0.50}       # healthy diet → less heartburn

# P(BP | HD) — BP: high blood pressure
P_BP_gHD = {True: 0.75, False: 0.15}

# P(CP | HD, HB)
P_CP_gHD_HB = {
    (True,  True):  0.95,   # HD + heartburn → chest pain very likely
    (True,  False): 0.80,
    (False, True):  0.40,
    (False, False): 0.05,
}

# =================================
# COMPUTE JOINT PROBABILITY
# P(X) = P(Ex)·P(Di)·P(HD|Ex,Di)·P(HB|Di)·P(BP|HD)·P(CP|HD,HB)
# =================================
def compute_joint(ex, di, hd, hb, bp, cp):
    p_ex = P_Ex[ex]
    p_di = P_Di[di]
    p_hd = P_HD_gEx_Di[(ex, di)] if hd else 1 - P_HD_gEx_Di[(ex, di)]
    p_hb = P_HB_gDi[di]          if hb else 1 - P_HB_gDi[di]
    p_bp = P_BP_gHD[hd]          if bp else 1 - P_BP_gHD[hd]
    p_cp = P_CP_gHD_HB[(hd, hb)] if cp else 1 - P_CP_gHD_HB[(hd, hb)]
    return p_ex * p_di * p_hd * p_hb * p_bp * p_cp

# Query: Observation: BP=True, CP=True → P(HD=True | BP=True, CP=True) = ?
print('\n📊 INFERENCE: P(HD | BP=True, CP=True)')
print('Marginalizing over all combinations...')

numerator, denominator = 0.0, 0.0
for ex in [True, False]:
    for di in [True, False]:
        for hb in [True, False]:
            p_hd_true  = compute_joint(ex, di, True,  hb, True, True)
            p_hd_false = compute_joint(ex, di, False, hb, True, True)
            numerator   += p_hd_true
            denominator += p_hd_true + p_hd_false

print(f'\n  P(HD=True  | BP=True, CP=True) = {numerator/denominator:.4f} ({numerator/denominator:.1%})')
print(f'  P(HD=False | BP=True, CP=True) = {1-numerator/denominator:.4f} ({1-numerator/denominator:.1%})')
print(f'\n  → "If BP is high and chest pain present, heart disease probability is {numerator/denominator*100:.1f}%"')


In [ ]:
# ============================================================
# BAYESIAN NETWORK — DAG STRUCTURE VISUALIZATION (networkx)
# ============================================================
try:
    import networkx as nx
    NETWORKX = True
except ImportError:
    NETWORKX = False

if NETWORKX:
    G = nx.DiGraph()
    edges = [('Exercise',       'Heart Disease'),
             ('Diet',           'Heart Disease'),
             ('Diet',           'Heartburn'),
             ('Heart Disease',  'Blood Pressure'),
             ('Heart Disease',  'Chest Pain'),
             ('Heartburn',      'Chest Pain')]
    G.add_edges_from(edges)

    pos = {'Exercise':      (0, 2), 'Diet':          (3, 2),
           'Heart Disease': (1.2, 1), 'Heartburn':   (3, 1),
           'Blood Pressure':(0, 0), 'Chest Pain':    (2.2, 0)}

    root_nodes = ['Exercise', 'Diet']
    mid_nodes  = ['Heart Disease', 'Heartburn']
    leaf_nodes = ['Blood Pressure', 'Chest Pain']

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.set_title('Heart Disease Bayesian Network (DAG)', fontsize=13, fontweight='bold')

    nx.draw_networkx_nodes(G, pos, nodelist=root_nodes, node_color='#2ecc71', node_size=2500, ax=ax)
    nx.draw_networkx_nodes(G, pos, nodelist=mid_nodes,  node_color='#e74c3c', node_size=2500, ax=ax)
    nx.draw_networkx_nodes(G, pos, nodelist=leaf_nodes, node_color='#3498db', node_size=2500, ax=ax)
    nx.draw_networkx_labels(G, pos, font_size=8, font_weight='bold', ax=ax)
    nx.draw_networkx_edges(G, pos, arrows=True, arrowsize=20,
                           edge_color='#2c3e50', width=2,
                           connectionstyle='arc3,rad=0.05', ax=ax)

    # Legend
    patches = [mpatches.Patch(color='#2ecc71', label='Root (Prior)'),
               mpatches.Patch(color='#e74c3c', label='Intermediate Variable'),
               mpatches.Patch(color='#3498db', label='Leaf (Observed)')]
    ax.legend(handles=patches, loc='lower left', fontsize=9)

    ax.text(1.5, -0.5,
            'P(X) = P(Ex)·P(Di)·P(HD|Ex,Di)·P(HB|Di)·P(BP|HD)·P(CP|HD,HB)',
            ha='center', fontsize=9, style='italic',
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

    ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('NetworkX not installed — install with: !pip install networkx')
    print('\nHeart Disease DAG Structure:')
    print('Exercise ──┐')
    print('           ├──▶ Heart Disease ──▶ Blood Pressure')
    print('Diet ──────┘       └────────────▶ Chest Pain')
    print('  └──▶ Heartburn ───────────────▶ Chest Pain')

# Local Markov Property demonstration
print('\n=== LOCAL MARKOV PROPERTY ===')
print('For the Heart Disease node:')
print('  pa(HD)       = {Exercise, Diet}')
print('  NonDesc(HD)  = {}')
print('  → HD ⊥ {} | {Exercise, Diet}')
print()
print('For the Blood Pressure node:')
print('  pa(BP)       = {Heart Disease}')
print('  NonDesc(BP)  = {Exercise, Diet, Heartburn}')
print('  → BP ⊥ {Ex, Di, HB} | {Heart Disease}')


---
# SECTION 4: Ensemble Learning

## 📖 Theoretical Summary

### Binomial Error Analysis (From Slide)
$$e_{ens} = \\sum_{i=13}^{25} \\binom{25}{i} \\varepsilon^i (1-\\varepsilon)^{25-i} \\approx 0.06 \\ll \\varepsilon = 0.35$$

### Bias-Variance Decomposition
$$\\text{gen.error}(m) = \\underbrace{c_1 \\cdot \\text{noise}}_{\\text{irreducible}} + \\underbrace{\\text{bias}(m)}_{\\text{bias}} + \\underbrace{c_2 \\cdot \\text{variance}(m)}_{\\text{variance}}$$

### AdaBoost — Weight Update
$$\\alpha_t = \\frac{1}{2}\\ln\\frac{1-\\varepsilon_t}{\\varepsilon_t} \\qquad w_i^{(t+1)} = \\frac{w_i^{(t)}}{Z_t} \\exp(-\\alpha_t y_i C_t(x_i))$$


In [ ]:
# ============================================================
# BINOMIAL ERROR ANALYSIS — THE POWER OF ENSEMBLES
# ============================================================
from scipy.special import comb

def ensemble_error(eps, n):
    """n independent classifiers, each with error rate eps"""
    majority = n // 2 + 1
    return sum(comb(n, i, exact=True) * eps**i * (1-eps)**(n-i)
               for i in range(majority, n+1))

print('=== BINOMIAL ERROR ANALYSIS ===')
eps_test = 0.35
n_test   = 25
e_ens    = ensemble_error(eps_test, n_test)
print(f'ε = {eps_test}, n = {n_test}')
print(f'e_ens = {e_ens:.4f}  (Slide reference: ≈ 0.06)')
print(f'Reduction: {eps_test} → {e_ens:.4f} = {(1-e_ens/eps_test)*100:.1f}% improvement')

# Different epsilon and n values
eps_values = np.linspace(0.05, 0.49, 100)
n_values   = [5, 11, 25, 51, 101]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Binomial Error Analysis — The Power of Ensemble Learning', fontsize=13, fontweight='bold')

ax1 = axes[0]
for n in n_values:
    errors = [ensemble_error(eps, n) for eps in eps_values]
    ax1.plot(eps_values, errors, linewidth=2, label=f'n={n}')
ax1.plot(eps_values, eps_values, 'k--', linewidth=2, label='Single Classifier (ε)')
ax1.axvline(0.35, color='gray', linestyle=':', alpha=0.7)
ax1.axhline(0.06, color='gray', linestyle=':', alpha=0.7)
ax1.scatter([0.35], [e_ens], c='red', s=150, zorder=5,
            label=f'Slide: n=25, ε=0.35 → {e_ens:.3f}')
ax1.set_xlabel('Individual Error (ε)'); ax1.set_ylabel('Ensemble Error')
ax1.set_title('Effect of Ensemble Size'); ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

# Bias-Variance visualization
ax2 = axes[1]
complexity = np.linspace(0.1, 10, 200)
bias     = 1.0 / (complexity**1.2)
variance = 0.02 * complexity**1.5
total    = bias + variance
noise    = 0.1 * np.ones_like(complexity)

ax2.plot(complexity, bias,     'b-',           lw=2.5, label='Bias')
ax2.plot(complexity, variance, color='orange', lw=2.5, label='Variance')
ax2.plot(complexity, total,    'g-',           lw=2.5, label='Total Error')
ax2.axhline(0.1, color='red', linestyle='--', lw=1.5, label='Noise (Irreducible)')

opt_idx = np.argmin(total)
ax2.axvline(complexity[opt_idx], color='purple', linestyle=':', lw=2,
            label=f'Optimum ≈ {complexity[opt_idx]:.1f}')
ax2.scatter([complexity[opt_idx]], [total[opt_idx]], c='purple', s=200, zorder=5)

ax2.set_xlabel('Model Complexity'); ax2.set_ylabel('Error')
ax2.set_title('Bias–Variance Trade-off'); ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3); ax2.set_ylim(0, 1.5)

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# ADABOOST — STEP BY STEP FOLLOWING SLIDE LOGIC
# ============================================================

class AdaBoostManual:
    """From-scratch implementation of the AdaBoost algorithm described in the slides"""
    
    def __init__(self, n_estimators=10):
        self.T = n_estimators
        self.alphas  = []
        self.stumps  = []
        self.history = []
    
    def fit(self, X, y):
        N = len(y)
        w = np.ones(N) / N    # Step 1: Equal initial weights
        
        for t in range(self.T):
            # Step 2: Train weighted decision stump
            stump = DecisionTreeClassifier(max_depth=1)
            stump.fit(X, y, sample_weight=w)
            y_pred = stump.predict(X)
            
            # Weighted error rate
            wrong  = (y_pred != y).astype(float)
            eps_t  = np.dot(w, wrong)   # ε_t = Σ wⱼ·I(Cₜ≠yⱼ)
            
            if eps_t >= 0.5:
                print(f'  Round {t+1}: ε={eps_t:.3f} ≥ 0.5 → STOP!')
                break
            
            # Alpha: α_t = ½ ln((1-ε)/ε)
            alpha_t = 0.5 * np.log((1 - eps_t) / eps_t)
            
            # Weight update: wᵢ ← wᵢ·exp(-αₜ·yᵢ·Cₜ(xᵢ))
            w = w * np.exp(-alpha_t * y * y_pred)
            w /= w.sum()   # normalize (Zₜ)
            
            self.alphas.append(alpha_t)
            self.stumps.append(stump)
            self.history.append({'round': t+1, 'eps': eps_t, 'alpha': alpha_t,
                                  'acc': accuracy_score(y, y_pred)})
        
        return self
    
    def predict(self, X):
        # C*(x) = sign(Σ αₜ·Cₜ(x))
        total = np.zeros(len(X))
        for alpha, stump in zip(self.alphas, self.stumps):
            total += alpha * stump.predict(X)
        return np.sign(total)


# Apply
np.random.seed(42)
X_ada, y_ada = make_classification(n_samples=200, n_features=2, n_informative=2,
                                    n_redundant=0, random_state=42)
y_ada = np.where(y_ada == 0, -1, 1)
X_ada_tr, X_ada_te, y_ada_tr, y_ada_te = train_test_split(X_ada, y_ada, test_size=0.3, random_state=42)

ada_manual = AdaBoostManual(n_estimators=15)
ada_manual.fit(X_ada_tr, y_ada_tr)

print('=== ADABOOST — ROUND-BY-ROUND ANALYSIS ===')
print("{:<8} {:<12} {:<12} {}".format("Round", "Error (ε)", "Alpha (α)", "Stump Acc"))
print('─' * 45)
for h in ada_manual.history:
    print(f"{h['round']:<8} {h['eps']:<12.4f} {h['alpha']:<12.4f} {h['acc']:.4f}")

# Accuracy
y_pred_ada = ada_manual.predict(X_ada_te)
acc_manual = accuracy_score(y_ada_te, y_pred_ada)
print(f'\n🎯 AdaBoost Manual Test Accuracy: {acc_manual:.4f}')

# sklearn AdaBoost comparison
ada_sklearn = AdaBoostClassifier(n_estimators=50, random_state=42)
ada_sklearn.fit(X_ada_tr, y_ada_tr)
print(f'🎯 sklearn AdaBoost  Test Accuracy: {accuracy_score(y_ada_te, ada_sklearn.predict(X_ada_te)):.4f}')


In [ ]:
# ============================================================
# BAGGING vs BOOSTING vs SINGLE — COMPREHENSIVE COMPARISON
# ============================================================
print('=== BAGGING vs BOOSTING vs SINGLE MODEL ===')
print('Data: make_classification (500 samples, 2 features)')
print()

X_ens, y_ens = make_classification(n_samples=500, n_features=2, n_informative=2,
                                    n_redundant=0, random_state=42)
X_ens_tr, X_ens_te, y_ens_tr, y_ens_te = train_test_split(
    X_ens, y_ens, test_size=0.25, random_state=42)

models = {
    'Decision Stump':        DecisionTreeClassifier(max_depth=1),
    'Deep Decision Tree':    DecisionTreeClassifier(max_depth=None),
    'Bagging (k=10)':        BaggingClassifier(n_estimators=10, random_state=42),
    'Bagging (k=50)':        BaggingClassifier(n_estimators=50, random_state=42),
    'Random Forest (k=50)':  RandomForestClassifier(n_estimators=50, random_state=42),
    'AdaBoost (k=50)':       AdaBoostClassifier(n_estimators=50, random_state=42),
}

print("{:<30} {:<14} {:<12} {}".format("Model", "Train Acc", "Test Acc", "CV Acc (5-fold)"))
print('─' * 72)

results_ens = []
for name, model in models.items():
    model.fit(X_ens_tr, y_ens_tr)
    train_acc = accuracy_score(y_ens_tr, model.predict(X_ens_tr))
    test_acc  = accuracy_score(y_ens_te, model.predict(X_ens_te))
    cv_acc    = cross_val_score(model, X_ens, y_ens, cv=5).mean()
    results_ens.append({'Model': name, 'Train': train_acc,
                        'Test': test_acc, 'CV': cv_acc})
    print(f'{name:<30} {train_acc:.4f}         {test_acc:.4f}       {cv_acc:.4f}')

# Visualization
df_ens = pd.DataFrame(results_ens)
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(df_ens))
w = 0.28
ax.bar(x-w,  df_ens['Train'], w, label='Train', color='#3498db', alpha=0.85)
ax.bar(x,    df_ens['Test'],  w, label='Test',  color='#e74c3c', alpha=0.85)
ax.bar(x+w,  df_ens['CV'],    w, label='CV',    color='#2ecc71', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(df_ens['Model'], rotation=25, ha='right', fontsize=9)
ax.set_ylabel('Accuracy'); ax.set_ylim(0.7, 1.05)
ax.set_title('Ensemble Methods Comparison', fontsize=13, fontweight='bold')
ax.legend(); ax.grid(True, axis='y', alpha=0.3)
ax.axvline(1.5, color='gray', linestyle=':', alpha=0.5)
ax.text(0.5, 0.72, 'Single Models',    ha='center', fontsize=9, color='gray')
ax.text(3.5, 0.72, 'Ensemble Models',  ha='center', fontsize=9, color='gray')
plt.tight_layout()
plt.show()


---
# SECTION 5: Class Imbalance

## 📖 Theoretical Summary

**Accuracy Paradox:** 1% positive data → "Predict everything negative" model → **99% accuracy but zero positives caught!**

**SMOTE Algorithm:** 
$$x_{new} = x + \\lambda(x_k - x), \\quad \\lambda \\in [0,1]$$


In [ ]:
# ============================================================
# ACCURACY PARADOX — LIVE DEMONSTRATION
# ============================================================
from sklearn.dummy import DummyClassifier

# Create imbalanced dataset (1000 negative, 50 positive)
np.random.seed(42)
n_neg, n_pos = 1000, 50
X_imb = np.r_[np.random.randn(n_neg, 2),
              np.random.randn(n_pos, 2) + [2.5, 2.5]]
y_imb = np.r_[np.zeros(n_neg), np.ones(n_pos)]

print('=== ACCURACY PARADOX ===')
print(f'Negative (0): {n_neg} samples | Positive (1): {n_pos} samples')
print(f'Class ratio: {n_pos/(n_neg+n_pos)*100:.1f}% positive')

X_d_tr, X_d_te, y_d_tr, y_d_te = train_test_split(
    X_imb, y_imb, test_size=0.2, random_state=42, stratify=y_imb)

# 1. Always-negative model
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_d_tr, y_d_tr)
y_pred_dummy = dummy.predict(X_d_te)

# 2. Real model
rf_imb = RandomForestClassifier(n_estimators=50, random_state=42)
rf_imb.fit(X_d_tr, y_d_tr)
y_pred_rf = rf_imb.predict(X_d_te)

for name, y_pred in [('Always Negative (Trivial)', y_pred_dummy),
                      ('Random Forest', y_pred_rf)]:
    acc = accuracy_score(y_d_te, y_pred)
    tp  = np.sum((y_d_te==1) & (y_pred==1))
    fn  = np.sum((y_d_te==1) & (y_pred==0))
    fp  = np.sum((y_d_te==0) & (y_pred==1))
    tn  = np.sum((y_d_te==0) & (y_pred==0))
    tpr = tp / (tp + fn) if (tp+fn) > 0 else 0
    print(f'\n🔹 {name}')
    print(f'   Accuracy: {acc:.4f} ({acc:.1%}) ← MISLEADING!')
    print(f'   TP={tp}, FP={fp}, FN={fn}, TN={tn}')
    print(f'   Recall (Sensitivity): {tpr:.4f}  ← TRUE PERFORMANCE')

print('\n⚠️  "Always negative" model achieves 95%+ accuracy but TP=0!')


In [ ]:
# ============================================================
# SMOTE — FROM-SCRATCH IMPLEMENTATION
# Slide formula: x_new = x + λ(xk - x), λ ∈ [0,1]
# ============================================================

def smote_manual(X_minority, k=5, n_new=50):
    """From-scratch implementation of the SMOTE algorithm from the slides"""
    from sklearn.neighbors import NearestNeighbors
    nn = NearestNeighbors(n_neighbors=k+1)
    nn.fit(X_minority)
    
    synthetics = []
    for _ in range(n_new):
        # Pick a random minority sample
        idx = np.random.randint(0, len(X_minority))
        x   = X_minority[idx]
        
        # Find k nearest neighbors
        _, neighbors = nn.kneighbors([x])
        neighbor_idx = np.random.choice(neighbors[0][1:])  # exclude self
        x_k = X_minority[neighbor_idx]
        
        # Interpolation: x_new = x + λ(xk - x)
        lam   = np.random.random()
        x_new = x + lam * (x_k - x)
        synthetics.append(x_new)
    
    return np.array(synthetics)

# Minority samples
np.random.seed(42)
X_minority = np.random.randn(20, 2) * 0.5 + [1.5, 1.5]
X_majority = np.random.randn(200, 2)

# Apply SMOTE
X_synthetic = smote_manual(X_minority, k=5, n_new=80)

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('SMOTE: Synthetic Minority Over-sampling Technique', fontsize=13, fontweight='bold')

# Before
axes[0].scatter(X_majority[:,0], X_majority[:,1], c='#3498db', s=20, alpha=0.5, label=f'Negative ({len(X_majority)})')
axes[0].scatter(X_minority[:,0], X_minority[:,1], c='#e74c3c', s=80, label=f'Positive ({len(X_minority)})')
axes[0].set_title('Before: Imbalanced Data')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# SMOTE
axes[1].scatter(X_minority[:,0],  X_minority[:,1],  c='#e74c3c', s=80, label='Original (+)')
axes[1].scatter(X_synthetic[:,0], X_synthetic[:,1], c='#e67e22', s=40, marker='^',
                label=f'Synthetic ({len(X_synthetic)})')

# Example interpolation
x_sample = X_minority[3]
x_neighbor = X_minority[7]
for lam_ex in [0.25, 0.5, 0.75]:
    x_new = x_sample + lam_ex * (x_neighbor - x_sample)
    axes[1].scatter(*x_new, c='yellow', s=120, marker='*', edgecolors='black', zorder=8)
axes[1].annotate('', xy=x_neighbor, xytext=x_sample,
                 arrowprops=dict(arrowstyle='->', color='black', lw=2))
axes[1].text((x_sample[0]+x_neighbor[0])/2, (x_sample[1]+x_neighbor[1])/2+0.1,
             'λ=0.25, 0.5, 0.75\n★=Synthetic', fontsize=7, ha='center')

axes[1].set_title('SMOTE: Sampling via Interpolation')
axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)

# After
X_pos_total = np.r_[X_minority, X_synthetic]
axes[2].scatter(X_majority[:,0],  X_majority[:,1],  c='#3498db', s=20, alpha=0.5, label=f'Negative ({len(X_majority)})')
axes[2].scatter(X_pos_total[:,0], X_pos_total[:,1], c='#e74c3c', s=40,
                label=f'Positive ({len(X_pos_total)})', alpha=0.7)
axes[2].set_title(f'After: Balanced Data ({len(X_pos_total)}/{len(X_majority)})')
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print('💡 SMOTE: Generates samples within the convex hull of original minority examples')
print('💡 Use with caution for samples near decision boundaries!')


In [ ]:
# ============================================================
# IMBALANCE HANDLING METHODS COMPARISON
# ============================================================
from sklearn.utils import resample

print('=== IMBALANCE HANDLING METHODS COMPARISON ===')
print(f'Original: Negative={n_neg}, Positive={n_pos}')

X_tr2, X_te2, y_tr2, y_te2 = train_test_split(
    X_imb, y_imb, test_size=0.25, random_state=42, stratify=y_imb)

# 1. Undersampling
idx_neg = np.where(y_tr2==0)[0]
idx_pos = np.where(y_tr2==1)[0]
idx_neg_under = np.random.choice(idx_neg, size=len(idx_pos)*2, replace=False)
idx_under = np.r_[idx_neg_under, idx_pos]
X_under, y_under = X_tr2[idx_under], y_tr2[idx_under]

# 2. Oversampling (Duplication)
n_oversample = len(idx_neg)
idx_pos_over = np.random.choice(idx_pos, size=n_oversample, replace=True)
idx_over = np.r_[idx_neg, idx_pos_over]
X_over, y_over = X_tr2[idx_over], y_tr2[idx_over]

# 3. SMOTE
X_pos_train = X_tr2[idx_pos]
X_smote_new = smote_manual(X_pos_train, k=3, n_new=len(idx_neg)-len(idx_pos))
X_smote = np.r_[X_tr2, X_smote_new]
y_smote = np.r_[y_tr2, np.ones(len(X_smote_new))]

# 4. Class weight
cw_ratio = len(idx_neg) / len(idx_pos)

# Test all methods
methods = [
    ('Original (Imbalanced)',   X_tr2,   y_tr2,   {}),
    ('Undersampling',           X_under, y_under, {}),
    ('Oversampling (Duplicate)',X_over,  y_over,  {}),
    ('SMOTE',                   X_smote, y_smote, {}),
    ('Class Weight',            X_tr2,   y_tr2,   {'class_weight': {0:1, 1:cw_ratio}}),
]

print("\n{:<28} {:<8} {:<12} {:<10} {}".format("Method", "Acc", "Precision", "Recall", "F1"))
print('─' * 65)

for name, X_t, y_t, params in methods:
    rf = RandomForestClassifier(n_estimators=50, random_state=42, **params)
    rf.fit(X_t, y_t)
    y_p  = rf.predict(X_te2)
    acc  = accuracy_score(y_te2, y_p)
    prec = precision_score(y_te2, y_p, zero_division=0)
    rec  = recall_score(y_te2, y_p)
    f1   = f1_score(y_te2, y_p)
    print(f'{name:<28} {acc:.3f}    {prec:.3f}        {rec:.3f}      {f1:.3f}')


---
# SECTION 6: Performance Metrics

## 📖 Theoretical Summary

**Confusion Matrix:** TP, TN, FP, FN

$$\\text{Precision} = \\frac{TP}{TP+FP}, \\quad \\text{Recall} = \\frac{TP}{TP+FN}$$

$$F_1 = \\frac{2 \\cdot r \\cdot p}{r+p}, \\quad F_\\beta = \\frac{(\\beta^2+1)rp}{\\beta^2 p + r}$$

$$\\text{TPR} = \\frac{TP}{TP+FN}, \\quad \\text{FPR} = \\frac{FP}{FP+TN}$$


In [ ]:
# ============================================================
# SLIDE ACCURACY PARADOX NUMERICAL EXAMPLE
# TP=0, FP=0, FN=50, TN=4950 → 99% accuracy but zero catches
# ============================================================

def compute_metrics(tp, fp, fn, tn, name='Model'):
    total = tp + fp + fn + tn
    acc   = (tp + tn) / total
    prec  = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec   = tp / (tp + fn) if (tp + fn) > 0 else 0
    spec  = tn / (tn + fp) if (tn + fp) > 0 else 0
    f1    = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
    fpr   = fp / (fp + tn) if (fp + tn) > 0 else 0
    fnr   = fn / (fn + tp) if (fn + tp) > 0 else 0
    fdr   = fp / (tp + fp) if (tp + fp) > 0 else 0
    g     = np.sqrt(prec * rec) if (prec * rec) > 0 else 0
    print(f'\n{"="*50}')
    print(f'  {name}')
    print(f'{"="*50}')
    print(f'  Confusion Matrix: TP={tp}, FP={fp}, FN={fn}, TN={tn}')
    print(f'  ─────────────────────────────────────')
    print(f'  Accuracy              = {acc:.4f} ({acc:.1%})')
    print(f'  Precision             = {prec:.4f}')
    print(f'  Recall (Sensitivity)  = {rec:.4f}')
    print(f'  Specificity (TNR)     = {spec:.4f}')
    print(f'  F1 Score              = {f1:.4f}')
    print(f'  FPR                   = {fpr:.4f}')
    print(f'  FNR                   = {fnr:.4f}')
    print(f'  FDR                   = {fdr:.4f}')
    print(f'  G-mean = √(Rec×Prec)  = {g:.4f}')

# Slide Accuracy Paradox example
compute_metrics(0, 0, 50, 4950, 'Trivial Model (Always Negative) — Slide Demonstration')

# Good model example
compute_metrics(40, 15, 10, 4935, 'Good Model (Some Misses)')

# Perfect model
compute_metrics(50, 0, 0, 4950, 'Perfect Model')

# Fβ analysis
print('\n=== Fβ ANALYSIS ===')
print('Example: TP=40, FP=15, FN=10 for different β values')
tp_ex, fp_ex, fn_ex = 40, 15, 10
prec_ex = tp_ex / (tp_ex + fp_ex)
rec_ex  = tp_ex / (tp_ex + fn_ex)
print(f'Precision={prec_ex:.3f}, Recall={rec_ex:.3f}')
print()
for beta in [0.5, 1.0, 2.0, 3.0]:
    f_beta = (beta**2 + 1) * prec_ex * rec_ex / (beta**2 * prec_ex + rec_ex)
    weight = 'Precision-weighted' if beta<1 else ('Equal' if beta==1 else 'Recall-weighted')
    print(f'  β={beta}: F_β = {f_beta:.4f} ({weight})')


In [ ]:
# ============================================================
# CONFUSION MATRIX — INTERACTIVE VISUALIZATION
# ============================================================
from sklearn.metrics import ConfusionMatrixDisplay

# Compare 3 models on Breast Cancer dataset
cancer = load_breast_cancer()
X_c, y_c = cancer.data, cancer.target
X_c_tr, X_c_te, y_c_tr, y_c_te = train_test_split(
    X_c, y_c, test_size=0.25, random_state=42, stratify=y_c)

sc = StandardScaler()
X_c_tr_s = sc.fit_transform(X_c_tr)
X_c_te_s = sc.transform(X_c_te)

models_cm = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM (RBF)':     SVC(kernel='rbf', probability=True),
    'AdaBoost':      AdaBoostClassifier(n_estimators=100, random_state=42),
}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Confusion Matrix Comparison (Breast Cancer)', fontsize=13, fontweight='bold')

for ax, (name, model) in zip(axes, models_cm.items()):
    model.fit(X_c_tr_s, y_c_tr)
    y_p = model.predict(X_c_te_s)
    cm  = confusion_matrix(y_c_te, y_p)
    
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                   display_labels=['Malignant', 'Benign'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    acc = accuracy_score(y_c_te, y_p)
    f1  = f1_score(y_c_te, y_p)
    ax.set_title(f'{name}\nAcc={acc:.3f} | F1={f1:.3f}', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# ROC CURVE AND PR CURVE — SLIDE M1/M2 COMPARISON
# ============================================================

# ROC and PR curves for 3 models (Breast Cancer)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('ROC and PR Curve Comparison', fontsize=13, fontweight='bold')

colors_roc = {'Random Forest': '#2ecc71', 'SVM (RBF)': '#3498db', 'AdaBoost': '#e74c3c'}

for name, model in models_cm.items():
    if hasattr(model, 'predict_proba'):
        y_score = model.predict_proba(X_c_te_s)[:, 1]
    else:
        y_score = model.decision_function(X_c_te_s)
    
    # ROC
    fpr_arr, tpr_arr, _ = roc_curve(y_c_te, y_score)
    roc_auc = auc(fpr_arr, tpr_arr)
    axes[0].plot(fpr_arr, tpr_arr, lw=2, color=colors_roc[name],
                 label=f'{name} (AUC={roc_auc:.3f})')
    
    # PR
    prec_arr, rec_arr, _ = precision_recall_curve(y_c_te, y_score)
    pr_auc = auc(rec_arr, prec_arr)
    axes[1].plot(rec_arr, prec_arr, lw=2, color=colors_roc[name],
                 label=f'{name} (AUPRC={pr_auc:.3f})')

# ROC — reference lines
axes[0].plot([0,1],[0,1], 'k--', lw=1.5, label='Random (AUC=0.5)')
axes[0].scatter([0],[1], c='gold', s=200, zorder=5, marker='*', label='Perfect (0,1)')
axes[0].set_xlabel('FPR (False Positive Rate)', fontsize=11)
axes[0].set_ylabel('TPR (Recall)', fontsize=11)
axes[0].set_title('ROC Curve', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)
axes[0].set_xlim([-0.02, 1.02]); axes[0].set_ylim([-0.02, 1.05])

# PR — reference line
alpha_ratio = np.sum(y_c_te==1) / len(y_c_te)
axes[1].axhline(alpha_ratio, color='gray', linestyle='--', lw=1.5,
                label=f'Random (α={alpha_ratio:.2f})')
axes[1].scatter([1],[1], c='gold', s=200, zorder=5, marker='*', label='Perfect (1,1)')
axes[1].set_xlabel('Recall', fontsize=11)
axes[1].set_ylabel('Precision', fontsize=11)
axes[1].set_title("PR Curve\n(More informative than ROC for imbalanced data)", fontsize=12, fontweight='bold')
axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)
axes[1].set_xlim([-0.02, 1.02]); axes[1].set_ylim([-0.02, 1.05])

plt.tight_layout()
plt.show()

print('\n💡 ROC: TPR vs FPR — relatively insensitive to class skew')
print('💡 PR : Precision vs Recall — more informative for imbalanced data')
print('💡 Larger AUC = better ranking quality!')


In [ ]:
# ============================================================
# OPTIMAL DECISION THRESHOLD — SLIDE LOGIC
# Adjust threshold to maximize F1
# ============================================================
rf_model = models_cm['Random Forest']
y_score_rf = rf_model.predict_proba(X_c_te_s)[:, 1]

thresholds = np.linspace(0.01, 0.99, 200)
f1_list    = []
acc_list   = []
prec_list  = []
rec_list   = []

for s in thresholds:
    y_th = (y_score_rf >= s).astype(int)
    f1_list.append(f1_score(y_c_te, y_th, zero_division=0))
    acc_list.append(accuracy_score(y_c_te, y_th))
    prec_list.append(precision_score(y_c_te, y_th, zero_division=0))
    rec_list.append(recall_score(y_c_te, y_th))

best_threshold = thresholds[np.argmax(f1_list)]
best_f1        = max(f1_list)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresholds, f1_list,   'b-', lw=2.5, label='F1')
ax.plot(thresholds, acc_list,  'g-', lw=2,   label='Accuracy')
ax.plot(thresholds, prec_list, 'r-', lw=2,   label='Precision')
ax.plot(thresholds, rec_list,  'm-', lw=2,   label='Recall')
ax.axvline(best_threshold, color='orange', linestyle='--', lw=2,
           label=f'Optimal Threshold = {best_threshold:.3f}')
ax.axvline(0.5, color='gray', linestyle=':', lw=1.5, label='Default Threshold = 0.5')
ax.scatter([best_threshold], [best_f1], c='orange', s=200, zorder=6)
ax.set_xlabel('Decision Threshold (sT)', fontsize=12)
ax.set_ylabel('Metric Value', fontsize=12)
ax.set_title('Optimal Decision Threshold Selection (Random Forest, Breast Cancer)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
ax.text(best_threshold+0.02, best_f1-0.05,
        f'F1={best_f1:.3f}', fontsize=9, color='orange')

plt.tight_layout()
plt.show()

print(f'\n📌 Default threshold (0.5): F1 = {f1_list[np.argmin(abs(thresholds-0.5))]:.4f}')
print(f'📌 Optimal threshold ({best_threshold:.3f}): F1 = {best_f1:.4f}')
print(f'\n💡 Setting threshold to {best_threshold:.3f} improves F1!')
print('💡 Slide formula: s* = argmax_s E(s) — search on validation set')


In [ ]:
# ============================================================
# COST MATRIX — SLIDE CANCER EXAMPLE
# FN cost is 100x more than FP!
# ============================================================

print('=== COST MATRIX — SLIDE CANCER EXAMPLE ===')
print()
print('Cost Matrix:')
print('              Predict +   Predict -')
print('  Actual +     C(+,+)=0   C(+,-)=100  ← Missing a patient')
print('  Actual -     C(-,+)=1   C(-,-)=0    ← Unnecessary alarm')
print()

C = {('+','+'): 0, ('+','-'): 100, ('-','+'): 1, ('-','-'): 0}

# Threshold formula: s* = C(-,+) / (C(-,+) + C(+,-))
c_fp = C[('-','+')]
c_fn = C[('+','-')]
s_optimal_cost = c_fp / (c_fp + c_fn)
print(f'Optimal Threshold (Cost-Sensitive):')
print(f's* = C(-,+) / (C(-,+) + C(+,-)) = {c_fp} / ({c_fp}+{c_fn}) = {s_optimal_cost:.4f}')
print(f'→ Set threshold to {s_optimal_cost:.4f} to minimize FN errors!')

# Expected cost comparison
print('\n=== EXPECTED COST COMPARISON ===')
for s in [0.5, s_optimal_cost, 0.1]:
    y_th = (y_score_rf >= s).astype(int)
    tp_m = np.sum((y_c_te==1) & (y_th==1))
    fp_m = np.sum((y_c_te==0) & (y_th==1))
    fn_m = np.sum((y_c_te==1) & (y_th==0))
    tn_m = np.sum((y_c_te==0) & (y_th==0))
    exp_cost = C[('+','-')]*fn_m + C[('-','+')]*fp_m
    print(f'  s={s:.4f}: TP={tp_m}, FP={fp_m}, FN={fn_m}, TN={tn_m} '
          f'→ Cost = {c_fn}×{fn_m} + {c_fp}×{fp_m} = {exp_cost}')

print('\n💡 Cost-sensitive learning: deliberately balance FN/FP trade-off!')
print('💡 Cancer diagnosis: FN>FP → lower threshold → catch more patients')


---
# 📝 LECTURE SUMMARY — Comparison of All Topics

## Algorithm Selection Guide

| Situation | Recommended Method | Why? |
|-----------|-------------------|------|
| Noisy data | **Bagging / Random Forest** | Reduces variance |
| Simple model, high bias | **Boosting (AdaBoost)** | Reduces bias |
| Speed + parallelism | **Bagging** | Independent training |
| Highest accuracy | **Gradient Boosting** | Mathematical optimization |
| Class imbalance | **SMOTE + Class Weight** | Augment minority |
| Variable dependencies | **Bayesian Network** | Modeling with DAG |
| Non-linear boundary | **SVM (RBF Kernel) / MLP** | Kernel trick |

## Metric Selection Guide

| Situation | Metric |
|-----------|--------|
| Balanced data | Accuracy, F1 |
| Imbalanced data | **PR-AUC**, Recall, F_β |
| FN costly (cancer, fraud) | **Recall (TPR)**, F₂ |
| FP costly (spam filter) | **Precision**, F₀.₅ |
| Model ranking quality | **ROC-AUC** |


In [ ]:
# ============================================================
# COMPREHENSIVE COMPARISON: Summary of All Algorithms
# ============================================================
cancer_data = load_breast_cancer()
X_fin, y_fin = cancer_data.data, cancer_data.target
X_f_tr, X_f_te, y_f_tr, y_f_te = train_test_split(
    X_fin, y_fin, test_size=0.2, random_state=42, stratify=y_fin)

sc_fin = StandardScaler()
X_f_tr_s = sc_fin.fit_transform(X_f_tr)
X_f_te_s = sc_fin.transform(X_f_te)

final_models = {
    'Decision Stump':        DecisionTreeClassifier(max_depth=1),
    'SVM (Linear)':          SVC(kernel='linear', probability=True),
    'SVM (RBF)':             SVC(kernel='rbf', probability=True),
    'MLP (100,50)':          MLPClassifier(hidden_layer_sizes=(100,50), max_iter=500, random_state=42),
    'Naive Bayes':           GaussianNB(),
    'Bagging (k=50)':        BaggingClassifier(n_estimators=50, random_state=42),
    'Random Forest (50)':    RandomForestClassifier(n_estimators=50, random_state=42),
    'AdaBoost (50)':         AdaBoostClassifier(n_estimators=50, random_state=42),
}

print('=== COMPREHENSIVE COMPARISON OF ALL ALGORITHMS ===')
print('Dataset: Breast Cancer (569 samples, 30 features, 2 classes)')
print()
print("{:<25} {:<8} {:<8} {:<8} {:<8} {}".format("Model", "Acc", "Prec", "Rec", "F1", "ROC-AUC"))
print('─' * 65)

records = []
for name, model in final_models.items():
    model.fit(X_f_tr_s, y_f_tr)
    y_p  = model.predict(X_f_te_s)
    acc  = accuracy_score(y_f_te, y_p)
    prec = precision_score(y_f_te, y_p)
    rec  = recall_score(y_f_te, y_p)
    f1v  = f1_score(y_f_te, y_p)
    if hasattr(model, 'predict_proba'):
        fpr_r, tpr_r, _ = roc_curve(y_f_te, model.predict_proba(X_f_te_s)[:,1])
        roc_a = auc(fpr_r, tpr_r)
    else:
        roc_a = float('nan')
    records.append({'Model': name, 'Acc': acc, 'Prec': prec, 'Rec': rec, 'F1': f1v, 'AUC': roc_a})
    print(f'{name:<25} {acc:.3f}    {prec:.3f}    {rec:.3f}    {f1v:.3f}    {roc_a:.3f}')

df_final = pd.DataFrame(records)
best = df_final.nlargest(1, 'F1').iloc[0]
print(f'\n🏆 Best Model (by F1): {best["Model"]} (F1={best["F1"]:.4f})')
